In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sea

class Logistic:

  def __init__(self,learning_rate=0.01,max_iter=1000):

    self.learning_rate=learning_rate
    self.max_iter=max_iter
    self.theta=None

    if learning_rate<=0 or max_iter<=0:
      raise ValueError("Learnign_rate and max_iter should be positive.")

  def sigmoid(self,z):

      return 1/(1+np.exp(-z))

  def fit(self,X_train,y_train):

      X_train=np.asarray(X_train)
      y_train=np.asarray(y_train).reshape(-1,1)

      if X_train.shape[0]!=y_train.shape[0]:
          raise ValueError("X_train and y_train must be same sample.")

      self.mean=np.mean(X_train,axis=0)

      self.std=np.std(X_train,axis=0)

      self.std[self.std==0]=1


      X_train=(X_train-self.mean)/self.std

      X_train=np.hstack((np.ones((X_train.shape[0],1)),X_train))

      self.theta=np.zeros((X_train.shape[1],1))

      self.log_loss=[]

      for iter in range(self.max_iter):

        z=X_train @ self.theta

        p=self.sigmoid(z)

        p=np.clip(p,1e-15,1-1e-15)
        self.log_loss.append(-np.mean(y_train * np.log(p) + (1-y_train)*np.log(1-p)))

        self.theta-=(self.learning_rate * X_train.T @ (p-y_train))/(X_train.shape[0])

      return self

   # private method for reduce code redundancy

  def __predict_helper(self,X_test):

      X_test=np.asarray(X_test)

      if self.theta is None:

          raise ValueError("model in not fitted.")

      if X_test.ndim==1:
        X_test=X_test.reshape(1,-1)

      X_test=(X_test-self.mean)/self.std
      X_test=np.hstack((np.ones((X_test.shape[0],1)),X_test))
      return self.sigmoid(X_test @ self.theta)

  def predict(self,X_test):

      pred=self.__predict_helper(X_test)
      pred=np.where(pred>=0.5,1,0).ravel()
      return pred

  def loss_graph(self):

      sea.lineplot(x=np.arange(self.max_iter),y=self.log_loss)
      plt.title("cost vs iter")
      plt.xlabel("max_iter")
      plt.ylabel("log_loss")
      plt.show()

  def predict_proba(self,X_test):
    return self.__predict_helper(X_test)

  def confusion_matrix(self,X_test,y_test):

    tp,fp,fn,tn=[0]*4
    y_test=np.asarray(y_test)

    if X_test.shape[0]==y_test.shape[0]:
      pred=self.predict(X_test)
      for i ,j in zip(y_test,pred):

        if i==1 and j==1:
          tp+=1
        elif i==1 and j==0:
          fn+=1
        elif i==0 and j==0:
          tn+=1
        elif i==0 and j==1:
          fp+=1

        else:

          raise ValueError("wrong input")

      return np.array([tp,fp,fn,tn]).reshape(2,2)

    else:
      raise ValueError("X_test and y_test should be same length")

  def accuracy(self,X_test,y_test):

    tp,fp,fn,tn=self.confusion_matrix(X_test,y_test).ravel()
    return (tp+tn)/(tp+fp+fn+tn)

  def precision(self,X_test,y_test):

    tp,fp,fn,tn=self.confusion_matrix(X_test,y_test).ravel()

    if tp+fp==0:
      return 0.0
    else:
      return tp/(tp+fp)

  def recall(self,X_test,y_test):

    tp,fp,fn,tn=self.confusion_matrix(X_test,y_test).ravel()

    if tp+fn==0:
      return 0.0
    else:
      return tp/(tp+fn)

  def specificity(self,X_test,y_test):

    tp,fp,fn,tn=self.confusion_matrix(X_test,y_test).ravel()

    if tn+fp==0:
      return 0.0
    else:
      return tn/(tn+fp)

  def f1_score(self,X_test,y_test):

    precision=self.precision(X_test,y_test)
    recall=self.recall(X_test,y_test)

    if precision+recall==0:
      return 0.0
    else:
      return 2*(precision*recall)/(precision+recall)

def score(self,X_test,y_test):
  return self.accuracy(X_test,y_test)